In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from faker import Faker

# Inicializar Faker en español para mayor realismo corporativo
fake = Faker('es_ES')
random.seed(42)
np.random.seed(42)

TOTAL_CLIENTES = 2000
print(f"Iniciando la generación de datos reales para {TOTAL_CLIENTES} clientes...")

# =====================================================================
# 1. GENERACIÓN DE TABLA: DATOS_CLIENTES
# =====================================================================
sectores = ['Logística y Distribución', 'Fintech & Banca', 'Retail y Comercio', 'Salud & Pharma', 'Manufactura Avanzada', 'Educación / EdTech', 'Entretenimiento & Medios']
soportes = ['Básico', 'Premium']
pagos = ['Tarjeta de Crédito', 'Transferencia Bancaria (ACH)', 'PayPal Business']

clientes_list = []

for i in range(1, TOTAL_CLIENTES + 1):
    cliente_id = f"CLI-{i:04d}"
    sector = random.choice(sectores)
    antiguedad = int(np.random.exponential(scale=14) + 3) # Distribución exponencial: más clientes nuevos que antiguos
    antiguedad = min(antiguedad, 48) # Límite máximo de 4 años (48 meses)
    
    # El número de licencias depende un poco del sector
    if sector in ['Fintech & Banca', 'Manufactura Avanzada']:
        licencias = int(np.random.normal(loc=40, scale=12))
    else:
        licencias = int(np.random.normal(loc=15, scale=6))
    licencias = max(5, licencias) # Mínimo 5 licencias
    
    soporte = np.random.choice(soportes, p=[0.65, 0.35]) # 65% básico, 35% premium
    pago = random.choice(pagos)
    
    # --- Lógica Probabilística Real para determinar el Churn Oculto ---
    # Calculamos un score de riesgo basado en factores demográficos y contractuales
    score_riesgo = 0.05
    if antiguedad < 6: score_riesgo += 0.25         # Clientes en periodo de adopción fallan más
    if soporte == 'Básico': score_riesgo += 0.15     # Menos atención personalizada
    if licencias < 10: score_riesgo += 0.10          # Cuentas pequeñas tienen menos compromiso
    if sector == 'Retail y Comercio': score_riesgo += 0.08 # Sector con alta volatilidad
    
    # Introducir un 10% de ruido aleatorio (clientes que se van sin motivo aparente o viceversa)
    churn = 1 if random.random() < score_riesgo else 0
    
    clientes_list.append([cliente_id, sector, antiguedad, licencias, soporte, pago, churn])

df_clientes = pd.DataFrame(clientes_list, columns=[
    'cliente_id', 'sector_industrial', 'antiguedad_meses', 'licencias_contratadas', 'tipo_soporte', 'metodo_pago', 'churn'
])


# =====================================================================
# 2. GENERACIÓN DE TABLA: USO_PLATAFORMA (Historial de 2 meses)
# =====================================================================
uso_list = []

for idx, row in df_clientes.iterrows():
    c_id = row['cliente_id']
    is_churn = row['churn']
    
    # Mes 1: Abril 2026 (Comportamiento Base)
    logins_m1 = int(np.random.normal(loc=28, scale=6))
    tiempo_m1 = round(float(np.random.normal(loc=75.0, scale=15.0)), 2)
    reportes_m1 = int(logins_m1 * random.uniform(0.7, 1.2))
    
    uso_list.append([c_id, '2026-04', max(1, logins_m1), max(5.0, tiempo_m1), max(0, reportes_m1)])
    
    # Mes 2: Mayo 2026 (Aquí se debe notar el patrón de abandono)
    if is_churn == 1:
        # Caída drástica en el uso para el cliente que va a abandonar
        logins_m2 = int(logins_m1 * random.uniform(0.05, 0.35))
        tiempo_m2 = round(float(tiempo_m1 * random.uniform(0.1, 0.4)), 2)
        reportes_m2 = int(reportes_m1 * random.uniform(0.0, 0.2))
    else:
        # Cliente saludable: mantiene o varía levemente su uso
        logins_m2 = int(np.random.normal(loc=29, scale=5))
        tiempo_m2 = round(float(np.random.normal(loc=78.0, scale=12.0)), 2)
        reportes_m2 = int(logins_m2 * random.uniform(0.8, 1.3))
        
    uso_list.append([c_id, '2026-05', max(0, logins_m2), max(0.0, tiempo_m2), max(0, reportes_m2)])

df_uso = pd.DataFrame(uso_list, columns=[
    'cliente_id', 'mes_reportado', 'logins_mes', 'tiempo_uso_promedio_min', 'reportes_generados'
])


# =====================================================================
# 3. GENERACIÓN DE TABLA: SOPORTE_TICKETS
# =====================================================================
tickets_list = []
ticket_counter = 1

for idx, row in df_clientes.iterrows():
    c_id = row['cliente_id']
    is_churn = row['churn']
    
    # Determinar cuántos tickets abre el cliente en el último mes
    if is_churn == 1:
        # Clientes frustrados abren más tickets de error
        num_tickets = int(np.random.poisson(lam=4.5)) 
    else:
        # Clientes estables abren pocos tickets incidentales
        num_tickets = int(np.random.poisson(lam=1.1))
        
    for _ in range(num_tickets):
        ticket_id = f"TCK-{ticket_counter:05d}"
        ticket_counter += 1
        
        # Fecha aleatoria durante el mes de Mayo 2026
        fecha_creacion = (datetime(2026, 5, 1) + timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d')
        
        if is_churn == 1:
            # Tiempos de espera largos y malas calificaciones (fricción)
            tiempo_res = round(float(np.random.gamma(shape=5, scale=10)), 1) # Sesgado a la derecha (muchas horas)
            csat = np.random.choice([1, 2, 3], p=[0.45, 0.35, 0.20])
        else:
            # Respuestas rápidas y clientes satisfechos
            tiempo_res = round(float(np.random.exponential(scale=4)), 1) # Mayoría resueltos rápido
            csat = np.random.choice([4, 5], p=[0.30, 0.70])
            
        tickets_list.append([ticket_id, c_id, fecha_creacion, max(0.5, tiempo_res), csat])

df_tickets = pd.DataFrame(tickets_list, columns=[
    'ticket_id', 'cliente_id', 'fecha_creacion', 'tiempo_resolucion_horas', 'csat_score'
])

# =====================================================================
# 4. EXPORTAR LOS DATASETS A FORMATO CSV
# =====================================================================
df_clientes.to_csv('datos_clientes.csv', index=False)
df_uso.to_csv('uso_plataforma.csv', index=False)
df_tickets.to_csv('soporte_tickets.csv', index=False)

print("\n¡Proceso Finalizado con Éxito!")
print(f"-> 'datos_clientes.csv' exportado con {df_clientes.shape[0]} filas.")
print(f"   Tasa de Churn simulada: {round((df_clientes['churn'].sum() / TOTAL_CLIENTES) * 100, 2)}%")
print(f"-> 'uso_plataforma.csv' exportado con {df_uso.shape[0]} filas (historial cruzado).")
print(f"-> 'soporte_tickets.csv' exportado con {df_tickets.shape[0]} registros de interacciones.")